## Получение списка вакансий с hh.ru

In [ ]:
import requests
import time
import json
import pandas as pd
import os
from tqdm.auto import tqdm

def collect_big_dataset():
    base_url = "https://api.hh.ru/vacancies"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    }

    search_queries = [
        # Development & Engineering
        "Python developer", "Java developer", "Frontend React developer",
        "Data Scientist", "DevOps engineer", "QA Automation engineer",
        "Golang developer", "C++ developer", "Node.js developer",
        "iOS developer", "Android developer", "ML Engineer", "Data Engineer",
        "PHP Developer", "Cybersecurity engineer",

        # Management & Analytics
        "System Analyst", "Product Manager", "Project Manager", "SRE engineer",
        "UI/UX Designer",

        # Trash / Noise (для калибровки и отсечения шума)
        "Бариста", "Курьер", "Менеджер по продажам",
        "Официант", "Администратор в кофейню"
    ]

    all_vacancies = []
    seen_ids = set()

    # Если файл уже существует, загрузим ID, чтобы не качать повторно
    if os.path.exists("hh_dataset.json"):
        with open("hh_dataset.json", "r", encoding="utf-8") as f:
            old_data = json.load(f)
            all_vacancies = old_data
            seen_ids = {str(v['id']) for v in old_data}
            print(f"Загружено из кэша: {len(seen_ids)} вакансий")

    for query in tqdm(search_queries, desc="Общий прогресс"):
        # Цикл по страницам
        for page in tqdm(range(5), desc=f"Поиск: {query}", leave=False):
            params = {
                "text": query,
                "area": 113,
                "per_page": 50,
                "page": page,
                "search_field": "name"
            }

            try:
                res = requests.get(base_url, params=params, headers=headers)

                if res.status_code == 403:
                    tqdm.write("!!! Капча или бан (список). Прерываем запрос.")
                    break
                if res.status_code != 200:
                    continue

                items = res.json().get('items', [])
                if not items:
                    break

                # Цикл по каждой вакансии на странице
                for item in tqdm(items, desc="Детали вакансий", leave=False):
                    v_id = str(item['id'])
                    if v_id in seen_ids:
                        continue

                    v_res = requests.get(f"{base_url}/{v_id}", headers=headers)

                    if v_res.status_code == 200:
                        v = v_res.json()
                        all_vacancies.append({
                            "id": v_id,
                            "target_role": query,
                            "name": v.get("name"),
                            "experience": v.get("experience", {}).get("name"),
                            "salary_from": (v.get("salary") or {}).get("from"),
                            "salary_to": (v.get("salary") or {}).get("to"),
                            "salary_curr": (v.get("salary") or {}).get("currency"),
                            "key_skills": [s['name'] for s in v.get("key_skills", [])],
                            "description": v.get("description"),
                            "alternate_url": v.get("alternate_url")
                        })
                        seen_ids.add(v_id)

                    elif v_res.status_code == 403:
                        tqdm.write(f"\nЛовим 403 на вакансии {v_id}. Пауза 20 сек...")
                        time.sleep(20)

                    # Короткая пауза между детальными запросами (важно!)
                    time.sleep(0.3)

                # Сохраняем после каждой страницы (на всякий случай)
                with open("hh_dataset.json", "w", encoding="utf-8") as f:
                    json.dump(all_vacancies, f, ensure_ascii=False, indent=4)

            except Exception as e:
                tqdm.write(f"Ошибка на запросе {query}: {e}")
                continue

    # Финальный экспорт в CSV
    if all_vacancies:
        df = pd.DataFrame(all_vacancies)
        df['key_skills'] = df['key_skills'].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")
        df.to_csv("hh_dataset.csv", index=False, encoding='utf-8-sig')
        print(f"\nСбор завершен! Итого уникальных: {len(all_vacancies)}")

if __name__ == "__main__":
    collect_big_dataset()

## Очистка данных

In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Скрипт для подготовки вакансий к обучению модели матчинга.
- Убирает лишние поля
- Чистит HTML из описания
- Извлекает грейд из name/experience
- Сохраняет в новый CSV
"""

import pandas as pd
import re
from bs4 import BeautifulSoup
from pathlib import Path


def clean_text(html: str) -> str:
    """
    Удаляет HTML-теги и нормализует текст.
    """
    if pd.isna(html) or not isinstance(html, str):
        return ""

    # 1. Парсим HTML и извлекаем текст
    text = BeautifulSoup(html, 'html.parser').get_text()

    # 2. Нормализуем пробелы (множественные пробелы/переносы → один пробел)
    text = re.sub(r'\s+', ' ', text).strip()

    # 3. (Опционально) Убираем явный шум: ссылки, телефоны, email
    text = re.sub(r'https?://\S+|t\.me/\S+|www\.\S+', '', text)
    text = re.sub(r'\+7[\d\s\-\(\)]{10,}', '', text)
    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '', text)

    return text


def extract_grade(name: str, experience: str) -> str:
    """
    Извлекает грейд из полей name и experience.
    Возвращает: 'senior', 'middle', 'junior' или 'unknown'.
    """
    # Приводим к строке и нижнему регистру
    name_str = str(name).lower() if pd.notna(name) else ""
    exp_str = str(experience).lower() if pd.notna(experience) else ""

    # Приоритет 1: явные маркеры в названии вакансии
    if any(w in name_str for w in ['senior', 'lead', 'главный', 'ведущий', 'head of']):
        return 'senior'
    if any(w in name_str for w in ['middle', 'mid-level', 'mid level', 'средний']):
        return 'middle'
    if any(w in name_str for w in ['junior', 'jun', 'начинающий', 'стажер', 'trainee']):
        return 'junior'

    # Приоритет 2: парсинг опыта работы
    # Senior: 6+ лет
    if re.search(r'(более\s*6|от\s*6|6\+|более\s*5|от\s*5|5\+)', exp_str):
        return 'senior'
    # Middle: 1-5 лет
    if re.search(r'(от\s*1\s*до\s*3|от\s*2\s*до\s*5|1\s*-\s*5|2|3|4|5|около\s*[2-5])', exp_str):
        return 'middle'
    # Junior: нет опыта или до 1 года
    if re.search(r'(нет\s*опыта|стаж|0|менее\s*1|до\s*1|без\s*опыта)', exp_str):
        return 'junior'

    return 'unknown'

def format_salary(row):
    """Форматирует вилку зарплаты из salary_from / salary_to."""
    salary_from = row.get("salary_from")
    salary_to = row.get("salary_to")

    # Оба значения есть → вилка
    if salary_from and salary_to:
        return f"{int(salary_from)} - {int(salary_to)}"

    # Только от
    elif salary_from:
        return f"от {int(salary_from)}"

    # Только до
    elif salary_to:
        return f"до {int(salary_to)}"

    # Не указана
    else:
        return "Не указано"

def prepare_vacancies(input_path: str, output_path: str) -> pd.DataFrame:
    """
    Основная функция подготовки данных.

    Args:
        input_path: Путь к исходному CSV с вакансиями
        output_path: Путь для сохранения очищенного CSV

    Returns:
        DataFrame с подготовленными данными
    """
    print(f"📥 Читаем данные из {input_path}...")
    df = pd.read_csv(input_path, encoding='utf-8')
    print(f"   Загружено строк: {len(df)}")

    # === 1. Отбираем нужные поля ===
    keep_columns = ['id', 'target_role', 'name', 'experience', 'key_skills', 'description']
    # Проверяем, что все колонки есть в файле
    missing = set(keep_columns) - set(df.columns)
    if missing:
        raise ValueError(f"В файле отсутствуют колонки: {missing}")

    df = df[keep_columns].copy()

    # === 2. Чистим описание ===
    print("🧹 Чистим HTML из описания...")
    df['description_clean'] = df['description'].apply(clean_text)

    # === 3. Извлекаем грейд ===
    print("🎯 Извлекаем грейды...")
    df['grade'] = df.apply(lambda row: extract_grade(row['name'], row['experience']), axis=1)

    df['skills'] = df['key_skills'].fillna('').astype(str)

    # === 5. Формируем итоговое поле для модели ===
    # Объединяем навыки + описание (навыки в начале усиливают сигнал)
    print("📝 Формируем финальный текст вакансии...")
    def make_vacancy_text(row):
        parts = []
        if row['skills']:
            parts.append(f"Навыки: {row['skills']}.")
        if row['description_clean']:
            parts.append(row['description_clean'])
        return ' '.join(parts).strip()

    df['vacancy_text'] = df.apply(make_vacancy_text, axis=1)
    df["job_title"] = df["name"].copy()

    df["salary"] = df.apply(format_salary, axis=1)

    # === 6. Отбираем финальные колонки для сохранения ===
    output_columns = [
        'id',           # ID для отслеживания
        'target_role',  # Категория (Python developer, Manager и т.д.)
        'job_title',
        'salary',
        'experience',
        'grade',        # Извлеченный грейд
        'skills',  # Список навыков (для аналитики)
        'vacancy_text'  # Готовый текст для подачи в BERT
    ]

    df_output = df[output_columns].copy()
    df_output = df_output.sample(frac=1)  # каждый запуск — новый порядок

    # === 7. Статистика ===
    print("\n📊 Статистика по грейдам:")
    print(df_output['grade'].value_counts())

    print(f"\n📄 Пример очищенного текста (первая вакансия):")
    sample = df_output.iloc[0]['vacancy_text']
    print(sample[:300] + "..." if len(sample) > 300 else sample)

    # === 8. Сохраняем ===
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_output.to_csv(output_path, index=False, encoding='utf-8')
    print(f"\n✅ Сохранено {len(df_output)} строк в {output_path}")

    return df_output


if __name__ == '__main__':
    # Настройки путей
    INPUT_CSV = '/content/drive/MyDrive/auto-hh/raw_hh_dataset.csv'      # Твой исходный файл
    OUTPUT_CSV = '/content/drive/MyDrive/auto-hh/hh_dataset.csv'   # Куда сохранить результат

    # Запуск
    df_clean = prepare_vacancies(INPUT_CSV, OUTPUT_CSV)

    # (Опционально) Быстрая проверка: вакансии с unknown grade
    unknown = df_clean[df_clean['grade'] == 'unknown']
    if len(unknown) > 0:
        print(f"\n⚠️  Найдено {len(unknown)} вакансий с grade='unknown'")
        print("Примеры:")
        print(unknown[['name', 'experience']].head())

📥 Читаем данные из /content/drive/MyDrive/auto-hh/raw_hh_dataset.csv...
   Загружено строк: 4499
🧹 Чистим HTML из описания...
🎯 Извлекаем грейды...
📝 Формируем финальный текст вакансии...

📊 Статистика по грейдам:
grade
middle    2745
junior     881
senior     873
Name: count, dtype: int64

📄 Пример очищенного текста (первая вакансия):
НАШ СТЕК: Hadoop, GreenPlum, S3; Airflow, Spark, Kafka, Debezium; ClickHouse, Superset; ЧТО ТЕБЯ ЖДЕТ: Анализ имеющегося функционала хранилища данных для целей миграции бизнес-процессов; Анализ новых требований от заказчиков по задачам развития отчетности; Реализация изменений и тестирование на стор...

✅ Сохранено 4499 строк в /content/drive/MyDrive/auto-hh/hh_dataset.csv


In [4]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/auto-hh/hh_dataset.csv", encoding='utf-8')

df.columns

Index(['id', 'target_role', 'name', 'experience', 'grade', 'skills',
       'vacancy_text'],
      dtype='object')

## Случайная выборка для тестов

In [16]:
import pandas as pd

# Укажите ваш путь к файлу
path = '/content/drive/MyDrive/auto-hh/hh_dataset.csv'

# 1. Читаем CSV
df = pd.read_csv(path)
n = 20

df_small = df.sample(n=min(n, len(df)))

# 3. Сохраняем в новый файл
output_path = '/content/drive/MyDrive/auto-hh/hh_dataset_sample.csv'
df_small.to_csv(output_path, index=False)

print(f"Готово! {n} рандомных вакансий сохранены в: {output_path}")

Готово! 20 рандомных вакансий сохранены в: /content/drive/MyDrive/auto-hh/hh_dataset_sample.csv


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/auto-hh/hh_dataset_sample.csv")

df.target_role.tolist()

## Генерация релевантных резюме

In [ ]:
"""
Генерация синтетических резюме через OpenAI API.
Пропорция: 70% Strong Positive + 30% Good Positive (с шумом).
"""

import os
import json
import time
import random
import pandas as pd
from openai import OpenAI
from pathlib import Path
from datetime import datetime
from google.colab import userdata

# === КОНФИГУРАЦИЯ ===
INPUT_CSV = '/content/drive/MyDrive/auto-hh/hh_dataset.csv'       # Твои очищенные вакансии
OUTPUT_CSV = '/content/drive/MyDrive/auto-hh/resumes_generated.csv'    # Куда сохранять резюме
CHECKPOINT_CSV = '/content/drive/MyDrive/auto-hh/resumes_checkpoint.csv' # Точки сохранения (на всякий случай)

# groq
API_KEY = userdata.get("GROQ_API_KEY")
BASE_URL= "https://api.groq.com/openai/v1"
OPENAI_MODEL = "openai/gpt-oss-120b"

# sambanova
# API_KEY = userdata.get("SAMBANOVA_API_KEY")
# BASE_URL = "https://api.sambanova.ai/v1"
# OPENAI_MODEL = "gpt-oss-120b"
# OPENAI_MODEL = 'Qwen3-235B'
# пробовать обе, пока лимиты не кончатся)

MAX_TOKENS = 2048
TEMPERATURE = 0.8

# Пропорции генерации
STRONG_RATIO = 0.65  # 70% идеальных
GOOD_RATIO = 0.25    # 30% с мягким шумом
NOISE_RATIO = 0.1

# === ПРОМПТЫ ===

SYSTEM_PROMPT = """
Ты — генератор синтетических резюме для обучения ML-модели матчинга.
Твоя задача — создать реалистичное резюме кандидата на основе вакансии.

Правила расстановки зарплат:
IT High (DevOps, SRE, ML, Data, Go, Java, C++, Analyst):
Junior: 90–130k | Middle: 230–320k | Senior: 380–550k

IT Core (React, Python, Node, iOS/Android, QA Auto, PM)
Junior: 70–110k | Middle: 180–250k | Senior: 280–400k

IT Entry (PHP, UI/UX)
Junior: 60–90k  | Middle: 140–190k | Senior: 200–300k

Non-IT
Сервис (Бариста, Официант, Админ): 50–90k + чаевые
Курьер: 70–120k (сдельно)
Продажи: 100–250k (оклад + %)

В регионах зарплата на 20-30% меньше

Верни ТОЛЬКО JSON в следующей структуре (без markdown, без пояснений):
{
    "target_info": {
        "title": "Должность из вакансии",
        "area": "Город",
        "desired_salary": число,
        "currency": "RUB",
        "total_experience_months": число
    },
    "skills_block": {
        "skill_set": ["Навык1", "Навык2"],
        "about_me": "2-3 предложения о кандидате"
    },
    "experience": [
        {
            "company": "Название компании",
            "position": "Должность",
            "duration_months": число,
            "description": "Текст о задачах и достижениях"
        }
    ],
    "education": {
        "university": "Используй разные российские вузы",
        "graduation_year": 2010-2030
    }
}

Важно:
1. Навыки должны соответствовать вакансии.
2. Опыт работы должен подтверждать грейд (Junior < 2 лет, Middle 2-5 лет, Senior 5+ лет).
3. В описании опыта используй реальные технологии и метрики.
4. Используй разное количество мест работы в зависимости от грейда (сеньор - 3-4, миддл - 1-2, джун 0-1)
5. Язык резюме — русский.
6. Не пиши больше 10 скиллов
"""

def get_strong_prompt(vacancy_text, grade, skills):
    """Промпт для идеального кандидата (Strong Positive)."""
    return f"""
Сгенерируй резюме ИДЕАЛЬНОГО кандидата на вакансию.

Требования к резюме:
- Все ключевые навыки ({skills}) должны быть явно указаны.
- Опыт работы полностью соответствует грейду {grade}.
- Минимум общих фраз ("стрессоустойчивый", "коммуникабельный").
- Четкие метрики и достижения в описании опыта.
- Структура идеальная, без ошибок.

Вакансия:
{vacancy_text}
"""

def get_good_prompt(vacancy_text, grade, skills):
    return f"""Сгенерируй резюме ХОРОШЕГО, но РЕАЛИСТИЧНОГО кандидата.

Требования:
1. Навыки: {skills} + 2-3 soft-skill слова (коммуникабельный, ответственный).
2. В 30% случаев НЕ указывай точные метрики (пиши "улучшил производительность", а не "на 30%").
3. Используй иногда менее формальные формулировки: "помогал с", "участвовал в", "работал над".
4. В 20% случаев укажи вуз попроще или без года выпуска.
5. Допускай небольшие повторы или неполные предложения в описании опыта.
6. Пиши живым, человеческим языком

Вакансия:
{vacancy_text}
"""

def get_noise_prompt(vacancy_text, grade, skills):
    """Промпт для генерации реалистичного резюме с шумом."""
    return f"""
Сгенерируй резюме РЕАЛИСТИЧНОГО кандидата (не идеальное).

Требования к шуму:
1. Навыки: {skills} + 2-3 soft-skill (коммуникабельный, ответственный).
2. Допускай небольшие опечатки в тексте (1-2 на всё резюме).
3. Используй менее формальные формулировки: "делал", "помогал с", "занимался".
4. В метриках пиши размыто: "улучшил производительность", "сократил время", а не точные %.
5. Не указывай год выпуска в образовании (или укажи вуз попроще).
6. Структура может быть немного небрежной (повторы, неполные предложения).

Важно: сохрани JSON-структуру ответа, но текст внутри сделай "живым".

Вакансия:
{vacancy_text}
Грейд: {grade}
"""

# === ФУНКЦИИ ===

def parse_llm_response(response_text: str) -> dict:
    """Парсит JSON из ответа LLM, убирая markdown-обертку."""
    text = response_text.strip()
    # Убираем markdown кодовые блоки, если есть
    if text.startswith('```json'):
        text = text[7:]
    if text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    return json.loads(text.strip())

def flatten_resume(resume_json: dict, vacancy_id: int, grade: str) -> dict:
    """Превращает JSON резюме в плоскую строку для CSV."""
    target = resume_json.get('target_info', {})
    skills = resume_json.get('skills_block', {})
    education = resume_json.get('education', {})

    # Собираем текст опыта
    exp_parts = []
    for job in resume_json.get('experience', []):
        duration = job.get('duration_months', 0)
        y, m = duration // 12, duration % 12
        dur_str = f"{y} г." if y else ""
        dur_str += f" {m} мес." if m else ""
        dur_str = dur_str.strip() or "менее года"

        desc = job.get('description', '').replace('S: ','').replace('T: ','')
        desc = desc.replace('A: ','').replace('R: ','')

        exp_parts.append(f"— {job.get('position')} в {job.get('company')} ({dur_str}): {desc}")

    return {
        'vacancy_id': vacancy_id,
        'grade': grade,
        'job_title': target.get('title', ''),
        'location': target.get('area', ''),
        'salary_val': target.get('desired_salary'),
        'salary_curr': target.get('currency', ''),
        'skills': ', '.join(skills.get('skill_set', [])),
        'about_me': skills.get('about_me', ''),
        'exp_text': '\n'.join(exp_parts),
        'edu_uni': education.get('university', ''),
        'edu_year': education.get('graduation_year'),
    }

def generate_resume(client, vacancy_row, resume_type: str) -> dict:
    """Генерирует одно резюме через OpenAI API с повторными попытками."""
    vacancy_text = vacancy_row['vacancy_text']
    grade = vacancy_row['grade']
    skills = vacancy_row['skills']
    vacancy_id = vacancy_row['id']

    # Выбираем промпт
    if resume_type == 'strong':
        user_prompt = get_strong_prompt(vacancy_text, grade, skills)
    elif resume_type == 'good':
        user_prompt = get_good_prompt(vacancy_text, grade, skills)
    else:
        user_prompt = get_noise_prompt(vacancy_text, grade, skills)

    # Попытки запроса (на случай ошибок API)
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )

            resume_json = parse_llm_response(response.choices[0].message.content)
            return flatten_resume(resume_json, vacancy_id, grade)

        except Exception as e:
            print(f"   ⚠️ Ошибка (попытка {attempt + 1}): {e}")
            if attempt < 2:
                time.sleep(2 ** (attempt*1.5))  # Экспоненциальная задержка
            else:
                return None  # Возвращаем None после 3 неудач
    time.sleep(2)

def load_checkpoint():
    """Загружает уже сгенерированные резюме, чтобы не начинать сначала."""
    if Path(CHECKPOINT_CSV).exists():
        return pd.read_csv(CHECKPOINT_CSV)
    return pd.DataFrame()

def save_checkpoint(df):
    """Сохраняет промежуточный результат."""
    df.to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')

# === ОСНОВНОЙ СКРИПТ ===

def main():
    print("🚀 Запуск генерации резюме...")

    # Инициализация клиента
    client = OpenAI(
        base_url=BASE_URL,
        api_key=API_KEY,
    )

    # Чтение вакансий
    print(f"📥 Чтение вакансий из {INPUT_CSV}...")
    df_vacancies = pd.read_csv(INPUT_CSV)
    print(f"   Найдено вакансий: {len(df_vacancies)}")

    # Загрузка чекпоинта
    df_checkpoint = load_checkpoint()
    generated_ids = set(df_checkpoint['vacancy_id'].tolist()) if not df_checkpoint.empty else set()
    print(f"   Уже сгенерировано: {len(generated_ids)}")

    # Фильтруем только новые
    df_to_generate = df_vacancies[~df_vacancies['id'].isin(generated_ids)].reset_index(drop=True)
    print(f"   Осталось сгенерировать: {len(df_to_generate)}")

    # Определяем типы резюме (70/30)
    resume_types = ['strong'] * int(len(df_to_generate) * STRONG_RATIO) + \
                   ['good'] * int((len(df_to_generate) * GOOD_RATIO)) + \
                   ['noise'] * int((len(df_to_generate) * NOISE_RATIO))
    if len(resume_types) != len(df_to_generate):
        resume_types.extend(['good'] * abs(len(df_to_generate) - len(resume_types)))
    random.shuffle(resume_types)  # Перемешиваем, чтобы не шли подряд

    # Генерация
    new_resumes = []
    for idx, row in df_to_generate.iterrows():
        resume_type = resume_types[idx]

        print(f"[{len(generated_ids) + idx + 1}/{len(generated_ids) + len(df_to_generate)}] Генерация {resume_type} для вакансии {row['id']} (грейд: {row['grade']})...")

        resume_data = generate_resume(client, row, resume_type)

        if resume_data == None:
          break

        if resume_data:
            new_resumes.append(resume_data)
            print(f"   ✅ Успешно сохранено")
        else:
            print(f"   ❌ Пропущено (ошибка API)")

        # Сохраняем чекпоинт каждые 10 вакансий
        df_current = pd.DataFrame(new_resumes)
        df_combined = pd.concat([df_checkpoint, df_current], ignore_index=True)
        save_checkpoint(df_combined)

        # Задержка для rate limiting
        time.sleep(2)

    # Финальное сохранение
    if new_resumes:
        df_new = pd.DataFrame(new_resumes)
        df_final = pd.concat([df_checkpoint, df_new], ignore_index=True)
        df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f"\n✅ Готово! Сохранено {len(df_final)} резюме в {OUTPUT_CSV}")
    else:
        print("\n⚠️ Новые резюме не сгенерированы")

if __name__ == '__main__':
    main()

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/auto-hh/resumes_generated_sample.csv")

for i,row in df.iterrows():
  print(f"Строка {i+1}")
  for k,v in row.items():
    print(f"{k}: {v}")
  print(end=3*'\n')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/auto-hh/hh_dataset.csv')

print(f"📊 Всего резюме: {len(df)}")
print(f"📈 Распределение грейдов:\n{df['grade'].value_counts()}")
print(f"🔧 Доменов (target_role): {df['target_role'].nunique()}")

# Проверка: есть ли soft-skills в skills_csv?
soft_keywords = ['коммуникабельный', 'ответственный', 'стрессоустойчивый', 'команда', 'лидер']
df['has_soft_in_skills'] = df['skills_csv'].str.lower().str.contains('|'.join(soft_keywords), na=False)
print(f"🎯 Резюме с софтами в навыках: {df['has_soft_in_skills'].sum()} / {len(df)}")

# Проверка длины текстов
print(f"📏 Средняя длина exp_text: {df['exp_text'].str.len().mean():.0f} символов")
print(f"📏 Средняя длина skills_csv: {df['skills_csv'].str.len().mean():.0f} символов")

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/auto-hh/resumes_generated_sample.csv')

soft_keywords = ['коммуникабельный', 'ответственный', 'стрессоустойчивый', 'команда', 'лидер', 'инициатив']

# Проверяем skills_csv
df['soft_in_skills'] = df['skills_csv'].str.lower().str.contains('|'.join(soft_keywords), na=False)

# Проверяем about_me
df['soft_in_about'] = df['about_me'].str.lower().str.contains('|'.join(soft_keywords), na=False)

# Проверяем exp_text
df['soft_in_exp'] = df['exp_text'].str.lower().str.contains('|'.join(soft_keywords), na=False)

print("📊 Где встречаются soft-skills:")
print(f"   В skills_csv: {df['soft_in_skills'].sum()} / {len(df)}")
print(f"   В about_me:   {df['soft_in_about'].sum()} / {len(df)}")
print(f"   В exp_text:   {df['soft_in_exp'].sum()} / {len(df)}")
print(f"   Хотя бы в одном месте: {(df['soft_in_skills'] | df['soft_in_about'] | df['soft_in_exp']).sum()} / {len(df)}")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/auto-hh/hh_dataset.csv")

df.target_role.unique().tolist()